# 🧪 Feature Pipeline Exploration & Extraction

Notebook này được sử dụng để kiểm tra luồng `build_features` của dự án Optiver. Nó thực thi master pipeline, xử lý dữ liệu raw parquet thành một DataFrame duy nhất chứa toàn bộ các features (Base + Nearest Neighbors) và lưu xuống file dạng Feather để sử dụng trong quá trình Training.

In [1]:
!git clone https://github.com/vinhVVN/Realized-Volatility-Prediction.git

Cloning into 'Realized-Volatility-Prediction'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 61 (delta 12), reused 60 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 43.09 KiB | 3.59 MiB/s, done.
Resolving deltas: 100% (12/12), done.


### 1. Import Thư Viện & Setup

In [2]:
# # Cài đặt các thư viện cần thiết cho việc đọc parquet và lưu feather trên Colab
!pip install fastparquet pyarrow

from google.colab import drive
import os

# Kết nối với Google Drive
drive.mount('/content/drive')

# CHÚ Ý: Đổi tên 'Optiver_Project' thành tên thư mục em đã tạo trên Drive
PROJECT_ROOT = '/content/Realized-Volatility-Prediction'

# Chuyển thư mục làm việc của Colab vào đúng thư mục dự án
os.chdir(PROJECT_ROOT)
print(f"Đã chuyển thư mục làm việc đến: {os.getcwd()}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 23.6 MB/s eta 0:00:00
Mounted at /content/drive
Đã chuyển thư mục làm việc đến: /content/Realized-Volatility-Prediction


In [3]:
import os
import sys
import warnings
import pandas as pd
import numpy as np

# Tắt cảnh báo chia cho NaN hoặc All-NaN slice của Numpy
warnings.filterwarnings('ignore')

# Trỏ đường dẫn gốc về thư mục root dự án để import `src`
sys.path.append(os.path.abspath('..'))

from src.data.data_loader import DataBlock
from src.features.feature_pipeline import build_features

### 2. Load Base Train DataFrame
Đọc tập `train.csv` chứa `time_id`, `stock_id` và nhãn `target` (Realized Volatility tương lai).

In [6]:
DRIVE_DIR = '/content/drive/MyDrive/Finance Optiver Realized/optiver-realized-volatility-prediction'
RAW_DIR = DRIVE_DIR
DATA_DIR = './data'

TRAIN_CSV = os.path.join(DRIVE_DIR, 'train.csv')

print("Đang đọc dữ liệu train.csv...")
try:
    df_train = pd.read_csv(TRAIN_CSV)
    print(f"Kích thước ban đầu: {df_train.shape}")
    display(df_train.head())
except FileNotFoundError:
    print(f"LỖI: Chưa tìm thấy file {TRAIN_CSV}. Bạn cần tải và giải nén dữ liệu Kaggle vào thư mục này.")

Đang đọc dữ liệu train.csv...
Kích thước ban đầu: (428932, 3)


,stock_id,time_id,target
0,0,5,0.004136
1,0,11,0.001445
2,0,16,0.002168
3,0,31,0.002195
4,0,62,0.001747


### 3. Thực Thi Master Pipeline
Đoạn mã này sẽ kích hoạt `build_features` để tự động load Book, Trade Parquet files, xử lý song song và sinh ra hàng trăm đặc trưng (Feature Engineering).

In [ ]:
print("Bắt đầu khởi chạy Feature Pipeline...")

# Lưu ý: Mặc định sẽ sử dụng tất cả CPU core để tính toán.
# Việc tính K-NN Neighbors mất khá nhiều RAM. Hàm đã tối ưu bộ nhớ.
df_features = build_features(
    df_train=df_train,
    data_dir=RAW_DIR,
    block=DataBlock.TRAIN,
    config_path= PROJECT_ROOT + '/configs/feature_config.yaml'
)

Bắt đầu khởi chạy Feature Pipeline...


### 4. Kiểm Tra & Đánh Giá Dữ Liệu Đầu Ra

In [ ]:
print(f"Kích thước DataFrame cuối cùng (N_rows, N_features): {df_features.shape}")

# Đếm số lượng NaN. Sẽ luôn có một chút NaN do K-NN tìm Neighbors không đủ dữ liệu
# LightGBM xử lý được NaN nên không cần drop/impute trừ khi xài Neural Network.
missing_vals = df_features.isnull().sum().sum()
print(f"Tổng số giá trị NaN trong DataFrame: {missing_vals}")

# Hiển thị 5 dòng đầu với max columns
pd.set_option('display.max_columns', 100)
display(df_features.head())

### 5. Lưu Dữ Liệu Dưới Dạng Feather
Lưu vào file `.feather` nhanh hơn nhiều so với `.csv` và `.parquet` khi làm việc với Pandas, đồng thời bảo toàn hoàn toàn Datatype đã ép kiểu.

In [ ]:
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

save_path = os.path.join(PROCESSED_DIR, 'features_v2.feather')
print(f"Đang lưu features xuống: {save_path} ...")

# Index phải được reset thì mới lưu feather được
df_features.reset_index(drop=True).to_feather(save_path)

print("Lưu thành công! Sẵn sàng cho Mốc Training Model.")